In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine

# 1. Database Connection Configuration
DB_USER = "root"
DB_PASS = "aroot"
DB_HOST = "localhost"
DB_PORT = "3306"
DB_NAME = "ecommerce_db"

# Create SQLAlchemy connection engine
engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# 2. Path to local raw CSV folder
DATA_DIR = r"C:\Users\amank\Desktop\E-commerce Analysis" 

print("Connection engine established.")

# we have doen this to store the data permanetly as using numpy  and read csv out data would 
# have been stored only in the ram as temporary do we connected it with sql server

In [ ]:
# using os.path.join because we want to create a dynamic file using variable data_dir
file_path = os.path.join(DATA_DIR,"olist_orders_dataset.csv")
df_orders = pd.read_csv(file_path)
cols =["order_delivered_carrier_date",
       "order_purchase_timestamp",
       "order_approved_at",
       "order_delivered_customer_date",
       "order_estimated_delivery_date",
       ]

for col in cols:
    df_orders[col] = pd.to_datetime(df_orders[col], errors="coerce")
# using the for loop for iterating each column and convrting it intop datetime

In [ ]:
# data Engineering
df_orders["delivery_delay_days"] = (df_orders["order_delivered_customer_date"]-df_orders["order_estimated_delivery_date"]).dt.days
df_orders["delivery_time_days"] = (df_orders["order_delivered_customer_date"]-df_orders['order_purchase_timestamp']).dt.days

In [ ]:
# send to my sql server
df_orders.to_sql(name="olist_orders_dataset",con=engine, if_exists="replace",index=False)

In [ ]:
files_to_load = {
    "olist_customers_dataset.csv": "olist_customers_dataset",
    "olist_order_items_dataset.csv": "olist_order_items_dataset",
    "olist_products_dataset.csv": "olist_products_dataset",
    "olist_order_payments_dataset.csv": "olist_order_payments_dataset",
    "olist_order_reviews_dataset.csv": "olist_order_reviews_dataset",
    "olist_sellers_dataset.csv": "olist_sellers_dataset",
    "olist_geolocation_dataset.csv": "olist_geolocation_dataset",
    "product_category_name_translation.csv": "product_category_name_translation",
}


for file_name, table_name in files_to_load.items():
    file_path = os.path.join(DATA_DIR,file_name)
    df=pd.read_csv(file_path)
    df.columns=df.columns.str.lower().str.replace(".","_")
    df.to_sql(name=table_name, con=engine, if_exists="replace",index=False)
    print(table_name + " loaded successfully")


In [ ]:
# Cell 4: Validate Row Counts in MySQL
validation_query = """
SELECT 'olist_orders_dataset' AS table_name, COUNT(*) AS row_count FROM olist_orders_dataset
UNION ALL
SELECT 'olist_customers_dataset', COUNT(*) FROM olist_customers_dataset
UNION ALL
SELECT 'olist_order_items_dataset', COUNT(*) FROM olist_order_items_dataset
UNION ALL
SELECT 'olist_order_payments_dataset', COUNT(*) FROM olist_order_payments_dataset;
"""

df_validation = pd.read_sql(validation_query, con=engine)
df_validation